# SQLite 汇总数据对比分析

用途：
- 自动分类数据库中的 `订单数据` / `汇总数据` 表
- 按日期范围读取汇总数据
- 做多表横向对比与趋势分析

In [13]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
from IPython.display import HTML, display

plt.style.use('default')
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Hiragino Sans GB', 'Heiti SC', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 统一显示四位小数（仅影响展示，不改底层数据）
pd.options.display.float_format = '{:.4f}'.format

# Plotly 渲染器自适应：优先 notebook 内联，失败时回退 iframe
try:
    import nbformat
    pio.renderers.default = 'notebook_connected'
except Exception:
    pio.renderers.default = 'iframe_connected'

# 最稳兜底：直接渲染 HTML

def show_plotly(fig):
    try:
        fig.show()
    except Exception:
        display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

DB_PATH = '/Users/yy/.openclaw/workspace/db/orders.db'
conn = sqlite3.connect(DB_PATH)
print('Connected:', DB_PATH)
print('Plotly renderer:', pio.renderers.default)

Connected: /Users/yy/.openclaw/workspace/db/orders.db
Plotly renderer: notebook_connected


In [14]:
# 1) 表分类：订单数据 vs 汇总数据
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
all_names = tables['name'].tolist()

order_tables = [t for t in all_names if '订单数据' in t]
summary_tables = [t for t in all_names if '汇总' in t or t.startswith('跨所每日数据分析_')]

print('订单数据表数量:', len(order_tables))
print('汇总数据表数量:', len(summary_tables))
display(pd.DataFrame({'order_tables': pd.Series(order_tables)}))
display(pd.DataFrame({'summary_tables': pd.Series(summary_tables)}))

订单数据表数量: 6
汇总数据表数量: 1


,order_tables
0,k1_订单数据_2026_03_27
1,k1_订单数据_2026_03_28
2,k1_订单数据_2026_03_29
3,k2_订单数据_2026_03_27
4,k2_订单数据_2026_03_28
5,k2_订单数据_2026_03_29


,summary_tables
0,跨所每日数据分析_汇总数据


In [15]:
# 2) 参数区（按需改）
START_DATE = '2026-03-01'
END_DATE = '2099-12-31'

# 只分析这一个汇总表
TARGET_SUMMARY_TABLE = '跨所每日数据分析_汇总数据'

# 在单表内做“字段对比”：按组配置你关心的指标
FIELD_GROUPS = {
    '盈亏对比': ['今日总盈亏（跨1）', '今日总盈亏（跨2）', '今日总盈亏（跨3）', '今日总盈亏(跨12)', '今日总盈亏（龟12）', '今日总盈亏（4-2）'],
    '交易量对比': ['总交易量（跨1）', '总交易量（跨2）', '总交易量（跨3）', '总交易量(跨12)', '总交易量（龟12）', '总交易量（4-2）'],
    '订单量对比': ['订单量（跨1）', '订单量（跨2）', '订单量（跨3）', '订单量(跨12)', '订单量（龟12）', '订单量（4-2）'],
    '捆绑费对比': ['捆绑费（跨1）', '捆绑费（跨2）', '捆绑费（跨3）', '捆绑费(跨12)', '捆绑费（龟12）', '捆绑费（4-2）'],
    'Gas费对比': ['今日总gas费（跨1）', '今日总gas费（跨2）', '今日总gas费（跨3）', '今日总gas费(跨12)', '今日总gas费（龟12）', '今日总gas费（4-2）'],
}

print('目标汇总表:', TARGET_SUMMARY_TABLE)
print('字段分组:', FIELD_GROUPS.keys())

目标汇总表: 跨所每日数据分析_汇总数据
字段分组: dict_keys(['盈亏对比', '交易量对比', '订单量对比', '捆绑费对比', 'Gas费对比'])


In [16]:
def clean_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype(str)
         .str.strip()
         .str.replace(',', '', regex=False)
         .str.replace('%', '', regex=False)
         .str.replace('U', '', regex=False)
         .replace({'': None, '-': None, '--': None, 'None': None, 'nan': None}),
        errors='coerce'
    )

In [17]:
# 3) 读取单一汇总表（日期过滤）
q = f'''
SELECT * FROM "{TARGET_SUMMARY_TABLE}"
WHERE DATE(日期) >= DATE('{START_DATE}') AND DATE(日期) <= DATE('{END_DATE}')
ORDER BY 日期
'''

df_all = pd.read_sql_query(q, conn)
df_all = df_all.dropna(how='all')
print('shape:', df_all.shape)
display(df_all.head())

shape: (48, 84)


,日期,跨所1,今日总盈亏（跨1）,总交易量（跨1）,订单量（跨1）,今日总gas费（跨1）,捆绑费（跨1）,bnb盈亏（跨1）,当日bnb价格,当日bnb价格变化（跨1）,...,捆绑费/成交量,捆绑费/gas费,gas/成交量,捆绑费/成交量（龟）,捆绑费/gas费（龟）,gas/成交量（龟）,捆绑费/成交量（4-2）,捆绑费/gas费（4-2）,gas/成交量（4-2）,今日总盈亏比（我/龟）
0,2026-03-01 00:00:00,跨所1,4014.7973,2420936.3047,8437.0000,41.9153,1096.7200,2048.0000,645.9300,645.9300,...,0.0003,17.7271,0.0000,0.0004,21.3635,0.0000,NaN,NaN,NaN,0.8882
1,2026-03-02 00:00:00,跨所1,4572.2475,2772829.9563,8761.0000,43.8638,1099.5400,2742.3575,630.5700,-15.3600,...,0.0003,17.4121,0.0000,0.0003,11.9603,0.0000,NaN,NaN,NaN,2.1628
2,2026-03-03 00:00:00,跨所1,-1153.4143,1621633.6196,5439.0000,26.9792,735.9700,-1910.0000,661.1600,30.5900,...,0.0003,12.5099,0.0000,0.0003,10.3880,0.0000,NaN,NaN,NaN,0.4782
3,2026-03-04 00:00:00,跨所1,3888.8739,8982216.7775,19441.0000,103.2270,1914.4500,3091.4520,652.8800,-8.2800,...,0.0002,15.3815,0.0000,0.0003,19.6161,0.0000,NaN,NaN,NaN,1.2967
4,2026-03-05 00:00:00,跨所1,1177.5137,10272791.0808,35443.0000,196.3210,1821.0700,-1399.2210,628.0100,-24.8700,...,0.0002,11.9313,0.0000,0.0003,16.5918,0.0000,NaN,NaN,NaN,0.1796


In [18]:
# 4) 关键指标列识别（来自字段分组）
metric_candidates = []
for _, cols in FIELD_GROUPS.items():
    metric_candidates.extend(cols)

metric_candidates = list(dict.fromkeys(metric_candidates))
metric_cols = [c for c in metric_candidates if c in df_all.columns]
print('识别到指标列:', metric_cols)

识别到指标列: ['今日总盈亏（跨1）', '今日总盈亏（跨2）', '今日总盈亏（跨3）', '今日总盈亏(跨12)', '今日总盈亏（龟12）', '今日总盈亏（4-2）', '总交易量（跨1）', '总交易量（跨2）', '总交易量（跨3）', '总交易量(跨12)', '总交易量（龟12）', '总交易量（4-2）', '订单量（跨1）', '订单量（跨2）', '订单量（跨3）', '订单量(跨12)', '订单量（龟12）', '订单量（4-2）', '捆绑费（跨1）', '捆绑费（跨2）', '捆绑费（跨3）', '捆绑费(跨12)', '捆绑费（龟12）', '捆绑费（4-2）', '今日总gas费（跨1）', '今日总gas费（跨2）', '今日总gas费（跨3）', '今日总gas费(跨12)', '今日总gas费（龟12）', '今日总gas费（4-2）']


In [19]:
# 5) 横向总量对比（先按列名分类，再按 valid_count 排序）
if not df_all.empty and metric_cols:
    rows = []
    for c in metric_cols:
        s = clean_numeric_series(df_all[c])
        rows.append({
            '列名': c,
            'valid_count': int(s.notna().sum()),
            'sum': float(s.sum(skipna=True)),
            'mean': float(s.mean(skipna=True)),
            'median': float(s.median(skipna=True)),
        })

    # 先按列名分类，再按 valid_count 排序（高 -> 低）
    wide_df = pd.DataFrame(rows).sort_values(['列名', 'valid_count'], ascending=[True, False])

    # 转成长表：一级分类=列名，二级分类=指标项
    cmp_df = wide_df.melt(
        id_vars=['列名', 'valid_count'],
        value_vars=['sum', 'mean', 'median'],
        var_name='二级分类',
        value_name='数值',
    ).sort_values(['列名', 'valid_count', '二级分类'], ascending=[True, False, True])

    display(cmp_df.round(6))
else:
    print('没有可对比的数据或指标列')

,列名,valid_count,二级分类,数值
30,今日总gas费(跨12),48,mean,308.0105
60,今日总gas费(跨12),48,median,363.6510
0,今日总gas费(跨12),48,sum,14784.5036
31,今日总gas费（4-2）,22,mean,281.5194
61,今日总gas费（4-2）,22,median,274.9343
...,...,...,...,...
88,订单量（跨3）,30,median,37887.5000
28,订单量（跨3）,30,sum,1094873.0000
59,订单量（龟12）,48,mean,36950.2292
89,订单量（龟12）,48,median,43437.5000


In [20]:
# 6) 趋势图（仅对比 跨12 vs 龟12，交互版）
TARGET_GROUP = '盈亏对比'  # 可选：'盈亏对比' / '交易量对比' / '订单量对比' / '捆绑费对比' / 'Gas费对比'
selected_metrics = [c for c in FIELD_GROUPS.get(TARGET_GROUP, []) if c in df_all.columns]

# 只保留跨12和龟12相关字段
selected_metrics = [c for c in selected_metrics if ('跨12' in c or '龟12' in c)]

print('TARGET_GROUP =', TARGET_GROUP)
print('selected_metrics =', selected_metrics)

if selected_metrics:
    p = df_all[['日期'] + selected_metrics].copy()
    p['日期'] = pd.to_datetime(p['日期'], errors='coerce')
    for c in selected_metrics:
        p[c] = clean_numeric_series(p[c])

    p = p.dropna(subset=['日期']).sort_values('日期')
    long_df = p.melt(id_vars='日期', value_vars=selected_metrics, var_name='metric', value_name='value').dropna()

    try:
        import plotly.express as px

        fig = px.line(
            long_df,
            x='日期',
            y='value',
            color='metric',
            markers=True,
            title=f'{TARGET_GROUP} 趋势对比（跨12 vs 龟12）',
            hover_data={'日期': True, 'value': ':.4f', 'metric': True},
        )
        fig.update_layout(hovermode='x unified', template='plotly_white')
        show_plotly(fig)
    except ModuleNotFoundError:
        print('未安装 plotly，请先运行: pip install plotly')
else:
    print('未找到跨12/龟12可画图字段')

TARGET_GROUP = 盈亏对比
selected_metrics = ['今日总盈亏(跨12)', '今日总盈亏（龟12）']


In [21]:
# 7) 组合指标对比（捆绑费率 / gas占比 / 盈亏效率）
# 说明：这里按“跨1 / 跨2 / 跨3 / 跨12 / 龟12 / 4-2”六组生成组合指标

metric_map = {
    '跨1': {'pnl': '今日总盈亏（跨1）', 'vol': '总交易量（跨1）', 'bundle': '捆绑费（跨1）', 'gas': '今日总gas费（跨1）'},
    '跨2': {'pnl': '今日总盈亏（跨2）', 'vol': '总交易量（跨2）', 'bundle': '捆绑费（跨2）', 'gas': '今日总gas费（跨2）'},
    '跨3': {'pnl': '今日总盈亏（跨3）', 'vol': '总交易量（跨3）', 'bundle': '捆绑费（跨3）', 'gas': '今日总gas费（跨3）'},
    '跨12': {'pnl': '今日总盈亏(跨12)', 'vol': '总交易量(跨12)', 'bundle': '捆绑费(跨12)', 'gas': '今日总gas费(跨12)'},
    '龟12': {'pnl': '今日总盈亏（龟12）', 'vol': '总交易量（龟12）', 'bundle': '捆绑费（龟12）', 'gas': '今日总gas费（龟12）'},
    '4-2': {'pnl': '今日总盈亏（4-2）', 'vol': '总交易量（4-2）', 'bundle': '捆绑费（4-2）', 'gas': '今日总gas费（4-2）'},
}

combo_rows = []
for k, m in metric_map.items():
    if all(col in df_all.columns for col in m.values()):
        pnl = clean_numeric_series(df_all[m['pnl']])
        vol = clean_numeric_series(df_all[m['vol']]).abs()
        bundle = clean_numeric_series(df_all[m['bundle']]).abs()
        gas = clean_numeric_series(df_all[m['gas']]).abs()

        bundle_rate = (bundle.sum() / vol.sum()) if vol.sum() else float('nan')
        gas_rate = (gas.sum() / vol.sum()) if vol.sum() else float('nan')
        cost_rate = ((bundle.sum() + gas.sum()) / vol.sum()) if vol.sum() else float('nan')
        pnl_eff = (pnl.sum() / vol.sum()) if vol.sum() else float('nan')

        combo_rows.append({
            'group': k,
            'pnl_sum': float(pnl.sum(skipna=True)),
            'vol_sum': float(vol.sum(skipna=True)),
            'bundle_sum': float(bundle.sum(skipna=True)),
            'gas_sum': float(gas.sum(skipna=True)),
            'bundle_rate': float(bundle_rate),
            'gas_rate': float(gas_rate),
            'cost_rate': float(cost_rate),
            'pnl_efficiency': float(pnl_eff),
        })

combo_df = pd.DataFrame(combo_rows)
display(combo_df.sort_values('pnl_efficiency', ascending=False).round(4))

,group,pnl_sum,vol_sum,bundle_sum,gas_sum,bundle_rate,gas_rate,cost_rate,pnl_efficiency
4,龟12,210734.5972,581526392.3400,266191.4145,11617.4633,0.0005,0.0000,0.0005,0.0004
5,4-2,115752.2010,343997327.2600,88059.3601,6193.4274,0.0003,0.0000,0.0003,0.0003
2,跨3,80364.7762,384151934.0488,104437.7910,6365.9576,0.0003,0.0000,0.0003,0.0002
0,跨1,88575.8553,457895969.9594,152857.9300,7165.7173,0.0003,0.0000,0.0003,0.0002
3,跨12,170586.1160,922046214.0330,264212.4900,14784.5036,0.0003,0.0000,0.0003,0.0002
1,跨2,82010.2607,464150244.0736,111354.5600,7618.7863,0.0002,0.0000,0.0003,0.0002


In [22]:
# 8) 组合指标趋势（仅跨12 vs 龟12，交互版）
# 可选：'bundle_rate' / 'gas_rate' / 'cost_rate' / 'pnl_efficiency'
TARGET_COMBO_METRIC = 'pnl_efficiency'
TARGET_GROUPS = ['跨12', '龟12']

trend_frames = []
for k, m in metric_map.items():
    if k not in TARGET_GROUPS:
        continue
    if all(col in df_all.columns for col in m.values()):
        d = df_all[['日期', m['pnl'], m['vol'], m['bundle'], m['gas']]].copy()
        d = d.rename(columns={m['pnl']: 'pnl', m['vol']: 'vol', m['bundle']: 'bundle', m['gas']: 'gas'})
        d['日期'] = pd.to_datetime(d['日期'], errors='coerce')
        d['pnl'] = clean_numeric_series(d['pnl'])
        d['vol'] = clean_numeric_series(d['vol']).abs()
        d['bundle'] = clean_numeric_series(d['bundle']).abs()
        d['gas'] = clean_numeric_series(d['gas']).abs()
        d = d.dropna(subset=['日期'])

        d['bundle_rate'] = d['bundle'] / d['vol'].replace(0, pd.NA)
        d['gas_rate'] = d['gas'] / d['vol'].replace(0, pd.NA)
        d['cost_rate'] = (d['bundle'] + d['gas']) / d['vol'].replace(0, pd.NA)
        d['pnl_efficiency'] = d['pnl'] / d['vol'].replace(0, pd.NA)
        d['group'] = k
        trend_frames.append(d[['日期', 'group', TARGET_COMBO_METRIC]])

trend_df = pd.concat(trend_frames, ignore_index=True) if trend_frames else pd.DataFrame()
if not trend_df.empty:
    trend_df = trend_df.dropna().sort_values('日期')
    try:
        import plotly.express as px

        fig = px.line(
            trend_df,
            x='日期',
            y=TARGET_COMBO_METRIC,
            color='group',
            markers=True,
            title=f'{TARGET_COMBO_METRIC} 趋势对比（跨12 vs 龟12）',
            hover_data={'日期': True, TARGET_COMBO_METRIC: ':.4f', 'group': True},
        )
        fig.update_layout(hovermode='x unified', template='plotly_white')
        show_plotly(fig)
    except ModuleNotFoundError:
        print('未安装 plotly，请先运行: pip install plotly')
else:
    print('组合指标趋势数据不足，无法绘图')

## 可视化增强（Summary Compare）

在现有表格基础上补充固定图版，便于日报快速浏览。

In [ ]:
# A) 六组合指标对比（静态柱状图）
if 'combo_df' in globals() and not combo_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.barplot(data=combo_df.sort_values('pnl_efficiency', ascending=False), x='group', y='pnl_efficiency', ax=axes[0])
    axes[0].set_title('各组 盈亏效率（pnl/vol）')
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].axhline(0, color='black', linewidth=0.8)

    sns.barplot(data=combo_df.sort_values('cost_rate', ascending=False), x='group', y='cost_rate', ax=axes[1])
    axes[1].set_title('各组 成本率（(bundle+gas)/vol）')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].axhline(0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.show()
else:
    print('请先运行组合指标对比单元（combo_df）。')

In [ ]:
# B) 指标相关性热力图（按目标指标组）
if 'selected_metrics' in globals() and selected_metrics and 'df_all' in globals() and not df_all.empty:
    p = df_all[selected_metrics].copy()
    for c in selected_metrics:
        p[c] = clean_numeric_series(p[c])
    p = p.dropna(how='all')

    if not p.empty and p.shape[1] >= 2:
        corr = p.corr(numeric_only=True)
        plt.figure(figsize=(max(8, 0.8 * len(corr.columns)), max(6, 0.7 * len(corr.columns))))
        sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False)
        plt.title(f'指标相关性热力图（{TARGET_GROUP}）')
        plt.tight_layout()
        plt.show()
    else:
        print('有效数据不足，无法绘制相关性热力图。')
else:
    print('请先运行“指标分组趋势”相关单元。')

In [23]:
conn.close()
print('SQLite connection closed.')

SQLite connection closed.
